# 02 &mdash; Four layers, one abstraction

> **Reading order.** These notebooks are a course, not a reference. Each one
> assumes the previous. `01` builds the tensor vocabulary, `02` builds the layers,
> `03` trains them on Cora, `04` proves they are correct and breaks them
> deliberately, `05` moves to graph-level tasks.
>
> Everything imports the local `gnn_lab` package -- edit the package and re-run
> the cell. The self-contained Kaggle run lives in `kaggle/`.

**Goal.** See that GCN, GraphSAGE, GAT and GIN are the *same function* with
three knobs set differently, and be able to write each one from the definition.

Every message-passing layer is:

```
1. MESSAGE      msg[e] = f(x_src[e], x_dst[e], edge_attr[e])
2. AGGREGATE    agg[i] = combine({ msg[e] : dst[e] == i })
3. UPDATE       x'[i]  = update(agg[i], x[i])
```

The four architectures differ only in `f`, `combine` and `update`:

| model | message `f` | combine | update |
|---|---|---|---|
| **GCN** | `W x_j`, scaled by `1/sqrt(d_i d_j)` | sum | add bias |
| **GraphSAGE** | `x_j` (identity) | mean / max / sum | `W_self x_i + W_neigh agg` |
| **GAT** | `W x_j`, scaled by learned `alpha_ij` | sum | add bias |
| **GIN** | `x_j` | sum | `MLP((1+eps) x_i + agg)` |

Once that table is obvious, "GNN architectures" stop being a zoo of papers.

---

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))     # so `import gnn_lab` works from notebooks/

import torch
import torch.nn as nn
import torch.nn.functional as F

from gnn_lab import (GCNScratch, GraphSAGEScratch, GATScratch, GINScratch,
                     GCNConvScratch, SAGEConvScratch, GATConvScratch, GINConvScratch,
                     load_cora, load_mutag, triangle_square_dataset, karate_club,
                     toy_graph, path_graph, two_communities,
                     train_node, train_node_multi_seed, train_graph_classification,
                     evaluate_node, set_seed, get_device, count_parameters,
                     accuracy, macro_f1, majority_baseline,
                     mean_pairwise_cosine, dirichlet_energy)
from gnn_lab.ops import (scatter_add, scatter_mean, scatter_max, scatter_softmax,
                         gcn_norm, add_self_loops, remove_self_loops, to_undirected,
                         degree, dense_to_edge_index)
from gnn_lab.layers.pooling import collate_graphs, global_pool, graph_batches
from gnn_lab import equivalence, viz

DEVICE = get_device()
RESULTS = os.path.join("..", "results")
os.makedirs(RESULTS, exist_ok=True)

print("torch  :", torch.__version__)
print("device :", DEVICE)
print("PyG    :", "available" if equivalence.PYG_AVAILABLE else f"missing ({equivalence.PYG_IMPORT_ERROR})")
set_seed(0)


## 1. GCN &mdash; Kipf & Welling, ICLR 2017

$$X' = \hat{D}^{-1/2}(A + I)\hat{D}^{-1/2}\, X\, W$$

Read right to left: a linear map, then a **degree-normalised weighted average
over the closed neighbourhood** (the `+I` is the self-loop).

Two design decisions are doing real work here:

**Why the self-loop.** Without `(A + I)` a node never sees its own features; it
can only be reconstructed from its neighbours. Ablation in notebook 04.

**Why the symmetric normalisation.** Plain averaging would be `D^{-1} A`. The
symmetric `D^{-1/2} A D^{-1/2}` splits the discount across both endpoints. That
is what stops hub nodes from dominating after several layers, and it is the
same operator behind a random walk with restart -- which is why GCN trains
stably rather than blowing up.

In [ ]:
from gnn_lab.layers.gcn import GCNConvScratch
import inspect

# the entire layer, so you can read it next to the maths
src = inspect.getsource(GCNConvScratch.forward)
print(src)

In [ ]:
g = toy_graph()
x, ei = g["x"].double(), g["edge_index"]

conv = GCNConvScratch(2, 2).double().eval()
with torch.no_grad():
    conv.lin.weight.copy_(torch.eye(2, dtype=torch.float64))
    conv.bias.zero_()
    out = conv(x, ei, 5)

expected = torch.tensor([[3,1],[2,2],[1,4],[3,2],[3,3]], dtype=torch.float64) / 3.0
print("GCN with W = identity, bias = 0:")
for i, (a, b) in enumerate(zip(out.tolist(), expected.tolist())):
    print(f"  node {i}: computed [{a[0]:.4f}, {a[1]:.4f}]   on paper [{b[0]:.4f}, {b[1]:.4f}]"
          f"   {'OK' if abs(a[0]-b[0])<1e-9 and abs(a[1]-b[1])<1e-9 else 'MISMATCH'}")

### The normalisation switch

`normalize=False` turns GCN into plain sum aggregation over the closed
neighbourhood. Watch the scale explode -- that is the practical reason the
`D^{-1/2}` term exists.

In [ ]:
norm_on  = GCNConvScratch(2, 2, normalize=True).double().eval()
norm_off = GCNConvScratch(2, 2, normalize=False).double().eval()
with torch.no_grad():
    for c in (norm_on, norm_off):
        c.lin.weight.copy_(torch.eye(2, dtype=torch.float64)); c.bias.zero_()
    a = norm_on(x, ei, 5)
    b = norm_off(x, ei, 5)
print(f"normalize=True   -> node values {a[0].tolist()[0]:.4f}, magnitude {float(a.norm()):.3f}")
print(f"normalize=False  -> node values {b[0].tolist()[0]:.4f}, magnitude {float(b.norm()):.3f}")
print(f"ratio            -> {float(b.norm() / a.norm()):.2f}x")
print()
print("On Cora the max degree is 169, so unnormalised sums would differ by two")
print("orders of magnitude between a hub and a leaf. Gradients cannot cope with that.")

## 2. GraphSAGE &mdash; Hamilton et al., NeurIPS 2017

$$h_i' = W_{\text{self}}\, h_i + W_{\text{neigh}}\, \text{AGG}\big(\{h_j : j \in N(i)\}\big)$$

The structural difference from GCN is small but consequential: **two independent
projections** instead of one, and an explicit aggregator choice. That buys two
things GCN cannot do:

**Inductive use.** Because aggregation runs over whatever neighbours are
present, a node never seen during training can still be embedded by running the
same forward pass on its own neighbourhood. GCN's `D^{-1/2}(A+I)D^{-1/2}` is
baked to one specific graph.

**Neighbour sampling.** You can subsample neighbour *sets*, so cost per node is
bounded and no longer scales with degree. That is what makes billion-edge graphs
trainable at all.

Note there are **no self-loops** here on purpose: `W_self` already handles the
node itself. Adding a loop *and* a root weight would double-count it, and
silently change what "mean" means.

In [ ]:
from gnn_lab.layers.sage import SAGEConvScratch

s = SAGEConvScratch(2, 2, aggr="mean").double().eval()
with torch.no_grad():
    s.lin_l.weight.copy_(torch.eye(2, dtype=torch.float64)); s.lin_l.bias.zero_()
    s.lin_r.weight.zero_()          # isolate the neighbour branch
    out = s(x, ei, 5)

hand = torch.tensor([[1.0, 0.5],    # mean(x1=[0,1], x3=[2,0])
                     [1.0, 0.5],    # mean(x0=[1,0], x2=[1,1])
                     [0.0, 1.5],    # mean(x1=[0,1], x4=[0,2])
                     [0.5, 1.0],    # mean(x0=[1,0], x4=[0,2])
                     [1.5, 0.5]],   # mean(x2=[1,1], x3=[2,0])
                    dtype=torch.float64)
print("SAGE mean aggregation, self branch zeroed:")
for i, (a, b) in enumerate(zip(out.tolist(), hand.tolist())):
    print(f"  node {i}: computed [{a[0]:.4f}, {a[1]:.4f}]   on paper [{b[0]:.4f}, {b[1]:.4f}]")
print("\nmatches:", bool(torch.allclose(out, hand, atol=1e-12)))
print("note: no self-loop, so node 0 sees only nodes 1 and 3 -- not itself.")

### The aggregator is the only thing that changes

Same layer, same weights, three `combine` operators. This is the message-passing
abstraction stripped down to its single degree of freedom.

In [ ]:
print("neighbour set of each node in the toy graph:")
for i in range(5):
    m = ei[1] == i
    print(f"  node {i} <- {sorted(set(ei[0][m].tolist()))}")

print()
print(f"{'node':>5} {'mean':>22} {'sum':>22} {'max':>22}")
outs = {}
for aggr in ("mean", "sum", "max"):
    c = SAGEConvScratch(2, 2, aggr=aggr).double().eval()
    with torch.no_grad():
        c.lin_l.weight.copy_(torch.eye(2, dtype=torch.float64)); c.lin_l.bias.zero_()
        c.lin_r.weight.zero_()
        outs[aggr] = c(x, ei, 5)

for i in range(5):
    row = " ".join(f"{str([round(v,2) for v in outs[a][i].tolist()]):>22}" for a in ("mean","sum","max"))
    print(f"{i:>5} {row}")

print()
print("sum is exactly degree x mean here, because the graph is 2-regular.")
print("On an irregular graph they diverge sharply -- sum lets hubs dominate.")

## 3. GAT &mdash; Velickovic et al., ICLR 2018

GCN's weights come from graph *structure*. GAT learns them from *features*:

$$e_{ij} = \text{LeakyReLU}\big(a^\top [W h_i \,\|\, W h_j]\big), \qquad
\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in N(i)} \exp(e_{ik})}$$

Four things worth getting exactly right:

**One shared projection `W`.** The paper transforms every node with the same `W`
and then scores pairs. PyTorch Geometric does the same when you pass a plain
`int` for `in_channels`; it only splits into `lin_src`/`lin_dst` for the
heterogeneous tuple case. Splitting them by accident doubles your parameters and
silently gives you a different model. `GATConvScratch` has a `separate_lin`
switch so you can measure the difference instead of guessing.

**The softmax is per-node.** Each node normalises over *its own* incoming edges.
That is a group-by, so it is `scatter_softmax` -- never `torch.softmax`.

**The score factorises.** `a^T[Wh_i || Wh_j]` splits into `a_s . Wh_j + a_d . Wh_i`:
one term depends only on the sender, one only on the receiver. You never form
the quadratic term, which would cost `O(E d^2)`.

**Dropout is applied to the attention coefficients**, not the features. That
regularises *which neighbours the model listens to* -- a structurally different
kind of noise.

In [ ]:
from gnn_lab.layers.gat import GATConvScratch

print("one shared projection by default (matches the paper and PyG):")
c = GATConvScratch(2, 4, heads=3)
print("  lin_src is lin_dst :", c.lin_src is c.lin_dst)
print("  state_dict keys    :", sorted(c.state_dict().keys()))
print()
print("and with separate_lin=True:")
c2 = GATConvScratch(2, 4, heads=3, separate_lin=True)
print("  lin_src is lin_dst :", c2.lin_src is c2.lin_dst)
print("  state_dict keys    :", sorted(c2.state_dict().keys()))
print("  extra parameters   :", sum(p.numel() for p in c2.parameters())
                            - sum(p.numel() for p in c.parameters()))

### Uniform attention collapses to GCN

Set both attention vectors to zero and every edge gets the same score, so the
softmax returns a uniform `1/|N(i)|`. On a *degree-regular* graph that is
exactly GCN's normalisation -- so GAT with zero attention must reproduce the GCN
output from section 1 bit for bit.

That is a strong check: two independently written layers agreeing on the same
hand-computed number.

In [ ]:
gat = GATConvScratch(2, 2, heads=1, concat=True).double().eval()
gcn = GCNConvScratch(2, 2).double().eval()
with torch.no_grad():
    gat.lin.weight.copy_(torch.eye(2, dtype=torch.float64))
    gat.att_src.zero_(); gat.att_dst.zero_(); gat.bias.zero_()
    gcn.lin.weight.copy_(torch.eye(2, dtype=torch.float64)); gcn.bias.zero_()

    a = gat(x, ei, 5)
    b = gcn(x, ei, 5)
    expected = torch.tensor([[3,1],[2,2],[1,4],[3,2],[3,3]], dtype=torch.float64) / 3.0

print("GAT (zero attention vectors) vs GCN vs the value computed on paper:")
for i in range(5):
    print(f"  node {i}: gat [{a[i,0]:.4f}, {a[i,1]:.4f}]   gcn [{b[i,0]:.4f}, {b[i,1]:.4f}]"
          f"   paper [{expected[i,0]:.4f}, {expected[i,1]:.4f}]")
print("\nall three agree:", bool(torch.allclose(a, b, atol=1e-12) and torch.allclose(a, expected, atol=1e-9)))

### With real attention weights the outputs diverge

Now give the attention vectors random values and watch each node weight its
neighbours differently. The returned `attention` matrix is `[E + N, heads]` --
one coefficient per edge per head, including the self-loops that were appended
inside `forward`.

Being able to *look at* these numbers is the main practical advantage of
writing the layer yourself.

In [ ]:
gat2 = GATConvScratch(2, 2, heads=4, concat=True, return_attention=True).double().eval()
with torch.no_grad():
    out, att = gat2(x, ei, 5)

print(f"attention shape: {tuple(att.shape)}  = ({ei.size(1)} edges + 5 self-loops, 4 heads)")
print()
print("per-node attention over incoming edges (head 0):")
dst_with_loops = torch.cat([ei[1], torch.arange(5)])
src_with_loops = torch.cat([ei[0], torch.arange(5)])
for i in range(5):
    m = dst_with_loops == i
    senders = src_with_loops[m].tolist()
    weights = [round(float(v), 4) for v in att[m, 0].tolist()]
    print(f"  node {i} <- {senders}  weights {weights}  sum {sum(weights):.4f}")

print()
print("every row sums to 1:",
      bool(torch.allclose(torch.zeros(5,4,dtype=torch.float64).index_add_(0, dst_with_loops, att),
                          torch.ones(5,4,dtype=torch.float64), atol=1e-10)))

In [ ]:
# output dimension follows the concat flag -- and so does the bias size,
# which is the detail most people get backwards
for concat in (True, False):
    c = GATConvScratch(2, 4, heads=3, concat=concat).double().eval()
    with torch.no_grad():
        o = c(x, ei, 5)
    print(f"concat={concat!s:5} -> output {tuple(o.shape)[1]:2d} channels, "
          f"bias has {c.bias.numel():2d} entries")
print()
print("concat=True  stacks the heads (3 x 4 = 12); concat=False averages them (4).")
print("The paper uses concat for hidden layers and averaging for the output layer.")

## 4. GIN &mdash; Xu et al., ICLR 2019

$$h_i' = \text{MLP}\Big((1 + \epsilon)\, h_i + \sum_{j \in N(i)} h_j\Big)$$

GIN exists because of an uncomfortable theorem: **mean and max aggregation
cannot distinguish certain non-isomorphic graphs.** Concretely, a node with one
neighbour of feature `1` and a node with two neighbours of feature `0.5` have
the same mean and the same max. Different structure, identical representation.

Sum aggregation can tell them apart, and the GIN paper shows sum + MLP is
exactly as expressive as the Weisfeiler-Lehman graph test -- the standard upper
bound for message-passing GNNs.

The `(1+eps)` term is a learnable dial between "count my neighbours" (`eps=0`)
and "ignore my neighbours" (`eps=-1`).

In practice: on Cora (node classification) GIN and GCN are within a point of
each other. On graph-level tasks the gap is real. Notebook 05 turns the theorem
into an experiment.

In [ ]:
from gnn_lab.layers.gin import GINConvScratch

gin = GINConvScratch(nn.Identity(), eps=0.0, train_eps=False).double().eval()
with torch.no_grad():
    out = gin(x, ei, 5)

hand = torch.tensor([[3,1],[2,2],[1,4],[3,2],[3,3]], dtype=torch.float64)   # x_i + sum(x_j)
print("GIN with eps=0 and MLP=Identity is a pure closed-neighbourhood sum:")
for i in range(5):
    print(f"  node {i}: computed {[round(v,3) for v in out[i].tolist()]}   "
          f"on paper {[round(v,3) for v in hand[i].tolist()]}")
print("\nmatches:", bool(torch.allclose(out, hand, atol=1e-12)))

gin2 = GINConvScratch(nn.Identity(), eps=2.0, train_eps=False).double().eval()
with torch.no_grad():
    out2 = gin2(x, ei, 5)
print(f"with eps=2 the self term is tripled: (1+2)*x_i + sum(x_j)")
print("  node 0:", [round(v,3) for v in out2[0].tolist()],
      " expected:", [round(v,3) for v in (3*x[0] + (hand[0]-x[0])).tolist()])
print("  matches:", bool(torch.allclose(out2, 3.0*x + (hand - x), atol=1e-12)))

## 5. Gradients actually flow

A layer that produces the right numbers in the forward pass can still be broken
in the backward pass -- a detached tensor somewhere silently zeroes a gradient.
Check every parameter of every layer receives a finite, non-trivial gradient.

In [ ]:
g = toy_graph()
xg, eig = g["x"].double(), g["edge_index"]

layers = {
    "GCN":  GCNConvScratch(2, 4),
    "SAGE": SAGEConvScratch(2, 4, aggr="mean"),
    "GAT":  GATConvScratch(2, 4, heads=2),
    "GIN":  GINConvScratch(nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 4))),
}
for name, layer in layers.items():
    layer = layer.double()
    layer.zero_grad(set_to_none=True)
    layer(xg, eig, 5).pow(2).sum().backward()
    bad = []
    for pname, p in layer.named_parameters():
        if p.grad is None:
            bad.append(f"{pname}: no gradient")
        elif not torch.isfinite(p.grad).all():
            bad.append(f"{pname}: non-finite gradient")
        elif float(p.grad.abs().sum()) == 0.0:
            bad.append(f"{pname}: zero gradient")
    n = sum(p.numel() for p in layer.parameters())
    print(f"  {name:5} {n:5d} parameters -> {'OK' if not bad else 'PROBLEM: ' + '; '.join(bad)}")

## 6. Stacking layers into models

A classifier is just a stack with ReLU and dropout between layers, and **no
activation on the final layer** (the loss expects logits).

In [ ]:
models = {
    "GCN":  GCNScratch(2, 8, 3, num_layers=2, dropout=0.0),
    "SAGE": GraphSAGEScratch(2, 8, 3, num_layers=2, dropout=0.0),
    "GAT":  GATScratch(2, 8, 3, heads=4, dropout=0.0),
}
for name, m in models.items():
    m = m.double().eval()
    with torch.no_grad():
        o = m(xg, eig)
    print(f"  {name:5} params {count_parameters(m):6d}   output {tuple(o.shape)}   "
          f"finite {bool(torch.isfinite(o).all())}")

print()
print("note how GAT has more parameters: 4 heads x 8 channels in the hidden layer.")
print("That is the standard trade -- attention costs parameters, and on Cora it")
print("buys roughly a point of accuracy.")

## 7. Summary

* Four architectures, one three-step template. The table at the top of this
  notebook is the thing to remember.
* **GCN** = degree-normalised mean over the closed neighbourhood.
* **GraphSAGE** = separate self and neighbour projections, pluggable aggregator,
  works inductively and supports sampling.
* **GAT** = the same, but the neighbour weights are learned per node and
  normalised with a *grouped* softmax.
* **GIN** = sum aggregation plus an MLP, which is provably more expressive than
  mean or max.
* Verify the backward pass, not just the forward pass.

**Next:** `03_node_classification_cora.ipynb` -- a real dataset, a real training
loop, and two data-loading gotchas that silently change your results.